# DarkPipe 0.8 — independent AION epoch

Reproduce la confirmación congelada de RID34056: 8 máximos de desarrollo y 0 confirmaciones holdout.

**Custodia:** descarga 564 MB solo en /content y los elimina al terminar. Exporta resultados compactos.

**Autoridad:** resultado de sensor en otra época de la misma familia, no instrumento independiente ni detección física.

In [ ]:
!git clone --depth 1 --branch v0.8.0 https://github.com/FacundoFirmenich/darkpipe-realdata.git
%cd darkpipe-realdata
!python -m pip install -q -e '.[test]'

In [ ]:
!python -m pytest -q

In [ ]:
import json
from pathlib import Path

campaign = Path('evidence/aion_independent_epoch_2026-08-25')
checked = json.loads((campaign / 'report.json').read_text())
assert checked['decision'] == 'NO_INDEPENDENT_HOLDOUT_CANDIDATE'
assert checked['confirmed_count'] == 0
assert len(checked['holdout_confirmation']) == 8
print('Recibo checked: 0/8; NO_INDEPENDENT_HOLDOUT_CANDIDATE')

In [ ]:
output = Path('/content/darkpipe_v08_reproduction')
scratch = Path('/content/darkpipe_v08_scratch')
candidate_commit = '2b4eba96bd813effcd6c4c0e0f165950b5a492ea'
!python run_darkpipe_aion_independent_search_v08.py --mode confirm --output {output} --scratch {scratch} --discovery {campaign / 'discovery.json'} --candidate-commit {candidate_commit}

In [ ]:
repeated = json.loads((output / 'report.json').read_text())
assert repeated['decision'] == checked['decision']
assert repeated['holdout_confirmation'] == checked['holdout_confirmation']
assert repeated['critical_max_statistic'] == checked['critical_max_statistic']
assert repeated['source']['sha256'] == checked['source']['sha256']
raw = scratch / checked['source']['filename']
assert not raw.exists()
print('Reproducción exacta y eliminación del bruto: OK')

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(output / 'holdout_confirmation.png')))
for row in repeated['holdout_confirmation']:
    print(row['candidate_id'], row['frequency_hz'], row['familywise_p'])

In [ ]:
import shutil
archive = shutil.make_archive(
    '/content/DarkPipe_AION_Independent_v08_results',
    'zip',
    output,
)
print(archive)